# INSY 684 Assignment 1: Event Streaming with Apache Kafka (Confluent Cloud) and Apache Spark (Databricks)

**Author:** Tasnim Hassan, McGill MMA

## Goal
Generate an event stream in Apache Kafka (Confluent Cloud) and process it with Apache Spark (Databricks), in both **batch** and **streaming** mode, using a new **Avro schema** registered in Confluent Schema Registry.

## Pipeline at a glance
```
                      Confluent Cloud                                   Databricks Free Edition (serverless Spark)
 +------------------------------------------------------+       +--------------------------------------------------------+
 | Schema Registry: subject databricks_test-value (Avro) | <---- | 2   register the CustomerEvent Avro schema               |
 |                                                      |       |                                                        |
 | Kafka topic: databricks_test (Basic cluster)         | <---- | 3.2 batch write (Spark DataFrame, sample data)           |
 |                                                      | <---- | 3.4 batch produce (confluent-kafka Python client)        |
 |                                                      | <---- | 4.2 streaming write (Structured Streaming, file source)  |
 |                                                      | ----> | 3.3 batch read + Avro decode                             |
 |                                                      | ----> | 4.3 streaming read -> Delta table (bronze) -> analytics  |
 +------------------------------------------------------+       +--------------------------------------------------------+
```

## Environment used
| Component | Version / setting |
|---|---|
| Kafka | Confluent Cloud, Basic cluster, topic `databricks_test` (6 partitions, default settings) |
| Schema management | Confluent Schema Registry (Stream Governance Essentials), Avro, subject `databricks_test-value` |
| Spark | Databricks Free Edition, serverless compute (Spark Connect) |
| Python client | `confluent-kafka[avro,schemaregistry]==2.16.1`, installed from PyPI |
| Storage | Unity Catalog volume `workspace.default.kafka_assignment` and Delta table `workspace.default.customer_events_bronze` |

## How the provided 2022 notebook was adapted
The original notebook was written for Databricks Community Edition in 2020 to 2022. Several parts no longer run as written, so each issue was identified and fixed:

| # | Original code | Problem today | Fix applied in this notebook |
|---|---|---|---|
| 1 | `bootstrap.servers = "..."` | Not valid Python: it tries to set an attribute on an undefined object `bootstrap`, so the cell raises `NameError`. | Renamed to a normal constant, `BOOTSTRAP_SERVERS`. |
| 2 | API key and secret typed inside every cell | Credentials end up in the notebook and in any export. The same 5 options were also copied 4 times. | Key and secret are read from a Databricks secret scope (the pattern used in the Week 1 course notebook). The options are built once in `KAFKA_OPTIONS`. |
| 3 | Databricks Community Edition | Community Edition was retired on 1 January 2026. Its replacement, Free Edition, offers serverless compute only. | Notebook targets Free Edition serverless compute. |
| 4 | `.writeStream ... .start()` with no trigger | Serverless compute rejects the default processing-time trigger with `INFINITE_STREAMING_TRIGGER_NOT_SUPPORTED`. | Every streaming query uses `.trigger(availableNow=True)` and `awaitTermination()`. |
| 5 | `checkpointLocation = "/dbfs/dir"` | DBFS root paths are a legacy pattern. Unity Catalog volumes are the supported location on Free Edition. | Checkpoints are stored under `/Volumes/workspace/default/kafka_assignment/...`. |
| 6 | Streaming job reads `databricks_test` and writes back to `databricks_test` | Every run copies the whole topic into itself, so the topic doubles on each execution (a feedback loop). | The streaming producer reads from a file source in a volume, and the streaming consumer writes to a Delta table. No loop. |
| 7 | `display(df_stream)` on a Kafka stream | `display()` on a streaming DataFrame starts a query with the default trigger, which serverless compute rejects. | The stream is written to a Delta table with `availableNow`, and the table is displayed. |
| 8 | Plain JSON strings in `value`, no schema | The assignment requires a new Avro schema registered in Confluent. | A `CustomerEvent` Avro schema is registered and all events use the Confluent Avro wire format. |
| 9 | `.option("subscribe", ...)` on the Kafka writer | `subscribe` is a reader option and is ignored by the writer. | The writer uses only `topic`. |

# 1. Setup

## 1.1. Install the Confluent Kafka client from PyPI
`confluent-kafka` is Confluent's official Python client. The `avro` and `schemaregistry` extras add the Schema Registry client and the Avro serializer. The version is pinned so the notebook gives the same result every time it runs.

In [ ]:
%pip install "confluent-kafka[avro,schemaregistry]==2.16.1" --quiet

In [ ]:
# Restart the Python process so the newly installed library can be imported.
dbutils.library.restartPython()

In [ ]:
import confluent_kafka

print("confluent-kafka (Python client):", confluent_kafka.version()[0])
print("librdkafka (C library)          :", confluent_kafka.libversion()[0])

## 1.2. Store the API keys as Databricks secrets (run once, then clear the values)
API keys must never be saved in the notebook. The cell below follows the same approach as the Week 1 course notebook (`Spark_Streaming_Kafka.ipynb`): the keys are stored once in a secret scope called `confluent`, and the rest of the notebook only reads them.

**How to use it:** uncomment the code, paste the four values, run the cell once, then put the placeholders back and comment the code again before exporting. Databricks also redacts secret values if they are ever printed.

In [ ]:
# from databricks.sdk import WorkspaceClient
#
# w = WorkspaceClient()
# if "confluent" not in {scope.name for scope in w.secrets.list_scopes()}:
#     w.secrets.create_scope(scope="confluent")
#
# secrets_to_store = {
#     "api_key": "<KAFKA_CLUSTER_API_KEY>",          # Confluent > Cluster > API keys
#     "api_secret": "<KAFKA_CLUSTER_API_SECRET>",
#     "sr_api_key": "<SCHEMA_REGISTRY_API_KEY>",     # Confluent > Environment > Schema Registry > API keys
#     "sr_api_secret": "<SCHEMA_REGISTRY_API_SECRET>",
# }
# for key, value in secrets_to_store.items():
#     w.secrets.put_secret(scope="confluent", key=key, string_value=value)
# print("Secrets saved:", [s.key for s in w.secrets.list_secrets(scope="confluent")])

## 1.3. Configuration
All settings live in this single cell. Endpoints and names are configuration and are shown in plain text. API keys come from the secret scope.

| Setting | Where to find it in Confluent Cloud |
|---|---|
| `BOOTSTRAP_SERVERS` | Cluster > Cluster settings > Endpoints > Bootstrap server |
| `SCHEMA_REGISTRY_URL` | Environment > Schema Registry (or the Stream Governance API panel) > Endpoint |
| `TOPIC_NAME` | Cluster > Topics |

In [ ]:
import json
import uuid
from datetime import datetime, timedelta, timezone
from pathlib import Path

from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.avro.functions import from_avro, to_avro
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)

# ---- Confluent Cloud (configuration, no credentials) ----
BOOTSTRAP_SERVERS = "<pkc-xxxxx.region.provider.confluent.cloud:9092>"
SCHEMA_REGISTRY_URL = "<https://psrc-xxxxx.region.provider.confluent.cloud>"
TOPIC_NAME = "databricks_test"
SCHEMA_SUBJECT = f"{TOPIC_NAME}-value"   # TopicNameStrategy: <topic>-value

# ---- Credentials (read from the Databricks secret scope, never typed here) ----
SECRET_SCOPE = "confluent"
KAFKA_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="api_key")
KAFKA_API_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE, key="api_secret")
SR_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="sr_api_key")
SR_API_SECRET = dbutils.secrets.get(scope=SECRET_SCOPE, key="sr_api_secret")

# ---- Databricks storage (Unity Catalog) ----
CATALOG, SCHEMA, VOLUME = "workspace", "default", "kafka_assignment"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.customer_events_bronze"

# A run id tags every event produced by this execution of the notebook (data lineage).
# It also keeps each run's landing folder and producer checkpoint separate, so the notebook can be re-run safely.
RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
LANDING_PATH = f"{VOLUME_PATH}/landing/{RUN_ID}"
PRODUCER_CHECKPOINT = f"{VOLUME_PATH}/checkpoints/stream_producer/{RUN_ID}"
CONSUMER_CHECKPOINT = f"{VOLUME_PATH}/checkpoints/stream_consumer_bronze"

# ---- Spark Kafka options, defined once and reused by every reader and writer ----
# The "kafkashaded." prefix is required on Databricks because the Kafka client classes are shaded.
JAAS_CONFIG = (
    "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required "
    f'username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";'
)
KAFKA_OPTIONS = {
    "kafka.bootstrap.servers": BOOTSTRAP_SERVERS,
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.sasl.jaas.config": JAAS_CONFIG,
}

# ---- confluent-kafka (librdkafka) client configuration ----
try:
    import certifi
    _ca_location = certifi.where()
except ImportError:
    _ca_location = "/etc/ssl/certs/ca-certificates.crt"

CLIENT_CONF = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanisms": "PLAIN",
    "sasl.username": KAFKA_API_KEY,
    "sasl.password": KAFKA_API_SECRET,
    "ssl.ca.location": _ca_location,
}
SR_CONF = {
    "url": SCHEMA_REGISTRY_URL,
    "basic.auth.user.info": f"{SR_API_KEY}:{SR_API_SECRET}",
}

# Create the Unity Catalog volume that holds checkpoints and the simulated event files.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

assert "<" not in BOOTSTRAP_SERVERS, "Replace BOOTSTRAP_SERVERS with your cluster's bootstrap server."
assert "<" not in SCHEMA_REGISTRY_URL, "Replace SCHEMA_REGISTRY_URL with your Schema Registry endpoint."
print("Run id          :", RUN_ID)
print("Topic           :", TOPIC_NAME)
print("Schema subject  :", SCHEMA_SUBJECT)
print("Volume          :", VOLUME_PATH)
print("Bronze table    :", BRONZE_TABLE)

## 1.4. Connectivity check
Before any Spark job runs, the Confluent `AdminClient` confirms three things: the credentials are accepted, the cluster is reachable from Databricks, and the topic exists. A failure here gives a clear message instead of a long Spark stack trace.

In [ ]:
from confluent_kafka.admin import AdminClient

admin = AdminClient(CLIENT_CONF)
cluster_metadata = admin.list_topics(timeout=20)

assert TOPIC_NAME in cluster_metadata.topics, (
    f"Topic '{TOPIC_NAME}' was not found. Create it in Confluent Cloud (Topics > Add topic) and re-run."
)
topic_metadata = cluster_metadata.topics[TOPIC_NAME]
print(f"Connected to cluster {cluster_metadata.cluster_id} ({len(cluster_metadata.brokers)} broker(s) visible).")
print(f"Topic '{TOPIC_NAME}' found with {len(topic_metadata.partitions)} partitions.")

# 2. Create a new Avro schema and register it in Confluent Schema Registry

The schema describes one **customer interaction event**. It acts as a data contract between the producers and consumers of the topic.

Design choices:
* `event_id` is a UUID, so consumers can detect duplicates (Kafka producers deliver *at least once*).
* `run_id` and `source` record lineage: which notebook run and which producer created each event.
* `event_type` is an **enum**, so a typo such as `PURCHSE` is rejected when the event is produced, not discovered later in a report.
* `amount` is **optional** (a union with `null`, default `null`), because only purchases have an amount. Optional fields with defaults also keep the schema compatible when it evolves.
* `event_time` uses the `timestamp-millis` logical type, so Spark reads it as a real timestamp.

Registration is idempotent: registering the same schema again returns the same schema id.

In [ ]:
CUSTOMER_EVENT_SCHEMA = {
    "type": "record",
    "name": "CustomerEvent",
    "namespace": "ca.mcgill.insy684.assignment1",
    "doc": "A customer interaction event produced to Confluent Cloud and processed in Databricks.",
    "fields": [
        {"name": "event_id", "type": "string", "doc": "Unique id (UUID) used for de-duplication."},
        {"name": "run_id", "type": "string", "doc": "Notebook run that produced the event."},
        {"name": "source", "type": "string", "doc": "Producer that created the event."},
        {"name": "customer_id", "type": "string"},
        {"name": "name", "type": "string"},
        {"name": "age", "type": "int"},
        {"name": "email", "type": "string"},
        {
            "name": "event_type",
            "type": {
                "type": "enum",
                "name": "EventType",
                "symbols": ["SIGNUP", "LOGIN", "PURCHASE", "LOGOUT"],
            },
        },
        {"name": "amount", "type": ["null", "double"], "default": None, "doc": "Purchase amount in CAD."},
        {"name": "event_time", "type": {"type": "long", "logicalType": "timestamp-millis"}},
    ],
}
AVRO_SCHEMA_STR = json.dumps(CUSTOMER_EVENT_SCHEMA)
EVENT_FIELDS = [field["name"] for field in CUSTOMER_EVENT_SCHEMA["fields"]]
REQUIRED_FIELDS = [f["name"] for f in CUSTOMER_EVENT_SCHEMA["fields"] if not isinstance(f["type"], list)]

print(json.dumps(CUSTOMER_EVENT_SCHEMA, indent=2))

In [ ]:
from confluent_kafka.schema_registry import Schema, SchemaRegistryClient

sr_client = SchemaRegistryClient(SR_CONF)
SCHEMA_ID = sr_client.register_schema(SCHEMA_SUBJECT, Schema(AVRO_SCHEMA_STR, schema_type="AVRO"))
latest = sr_client.get_latest_version(SCHEMA_SUBJECT)

print(f"Subject        : {SCHEMA_SUBJECT}")
print(f"Schema id      : {SCHEMA_ID}")
print(f"Latest version : {latest.version}")

### Confluent Avro wire format
Every message written by a Confluent Avro serializer has a 5 byte header followed by the Avro binary body:

| Byte(s) | Content |
|---|---|
| 0 | Magic byte, always `0` |
| 1 to 4 | Schema id from Schema Registry (big endian integer) |
| 5 onward | Record encoded in Avro binary format |

Spark's built-in `to_avro` and `from_avro` produce and read only the Avro body. The two helpers below add and remove the header, so messages written by Spark and by the Python client are identical in format. Because of that, Confluent Cloud's message browser can decode all of them, and Spark can read all of them.

In [ ]:
CONFLUENT_HEADER = bytearray(b"\x00" + SCHEMA_ID.to_bytes(4, byteorder="big"))


def to_confluent_avro(df):
    # Encode the event columns as a Confluent Avro message, keyed by customer_id.
    # Keying by customer_id sends all events of one customer to the same partition, which keeps them in order.
    return df.select(
        F.col("customer_id").alias("key"),
        F.concat(
            F.lit(CONFLUENT_HEADER),
            to_avro(F.struct(*[F.col(name) for name in EVENT_FIELDS]), AVRO_SCHEMA_STR),
        ).alias("value"),
    )


def from_confluent_avro(kafka_df):
    # Decode Kafka records written with this schema id. Records that do not use it are skipped.
    return (
        kafka_df
        .withColumn("magic_byte", F.hex(F.expr("substring(value, 1, 1)")))
        .withColumn("schema_id", F.conv(F.hex(F.expr("substring(value, 2, 4)")), 16, 10).cast("int"))
        .where((F.col("magic_byte") == "00") & (F.col("schema_id") == SCHEMA_ID))
        .withColumn(
            "event",
            from_avro(F.expr("substring(value, 6, length(value) - 5)"), AVRO_SCHEMA_STR, {"mode": "PERMISSIVE"}),
        )
        .where(F.col("event").isNotNull())
        .select(
            F.col("key").cast("string").alias("kafka_key"),
            "partition",
            "offset",
            F.col("timestamp").alias("kafka_timestamp"),
            "schema_id",
            "event.*",
        )
    )


def assert_no_nulls(df, columns, label):
    # Data quality gate: required Avro fields must never be null, otherwise to_avro fails at write time.
    null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in columns]).first().asDict()
    bad = {c: n for c, n in null_counts.items() if n}
    assert not bad, f"{label}: null values found in required fields {bad}"
    print(f"{label}: data quality check passed (no nulls in {len(columns)} required fields).")


print("Confluent header bytes:", CONFLUENT_HEADER.hex(" "))

# 3. Batch Processing

## 3.1. Load topic data from Confluent in batch mode
A batch read takes a bounded snapshot of the topic, from the earliest to the latest offset available when the query starts. The raw Kafka columns are `key`, `value` (bytes), `topic`, `partition`, `offset`, `timestamp` and `timestampType`.

In [ ]:
df_kafka = (
    spark.read.format("kafka")
    .options(**KAFKA_OPTIONS)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

print(f"Messages currently in '{TOPIC_NAME}': {df_kafka.count()}")
display(df_kafka)

## 3.2. Write a Kafka sink for batch queries
### 3.2.1. Create sample data
The three sample records from the provided notebook are kept as the input.

In [ ]:
# Create the sample data (same records as the provided notebook)
sample_data = Row("key", "value", "topic")
sample1 = sample_data('1', '{"name": "Jane Doe", "age": 29, "email": "jane.doe@example.com"}', "test-topic")
sample2 = sample_data('2', '{"name": "Fatih", "age": 24, "email": "fatih@example.com"}', "test-topic")
sample3 = sample_data('3', '{"name": "John Doe", "age": 35, "email": "john.doe@example.com"}', "test-topic")

print(sample1)

### 3.2.2. Create a DataFrame from the sample data and shape it to the Avro schema
The JSON in `value` is parsed into columns. Each customer then becomes a `SIGNUP` event that matches the `CustomerEvent` schema.

In [ ]:
df = spark.createDataFrame([sample1, sample2, sample3])
df.show(truncate=False)

In [ ]:
profile_schema = "name STRING, age INT, email STRING"

batch_events = (
    df.withColumn("profile", F.from_json("value", profile_schema))
    .select(
        F.expr("uuid()").alias("event_id"),
        F.lit(RUN_ID).alias("run_id"),
        F.lit("spark-batch").alias("source"),
        F.format_string("C%03d", F.col("key").cast("int")).alias("customer_id"),
        F.col("profile.name").alias("name"),
        F.col("profile.age").alias("age"),
        F.col("profile.email").alias("email"),
        F.lit("SIGNUP").alias("event_type"),
        F.lit(None).cast("double").alias("amount"),
        F.current_timestamp().alias("event_time"),
    )
)

assert_no_nulls(batch_events, REQUIRED_FIELDS, "Batch events")
display(batch_events)

### 3.2.3. Write data from the DataFrame to the Confluent Kafka topic

In [ ]:
batch_payload = to_confluent_avro(batch_events)

(
    batch_payload.write.format("kafka")
    .options(**KAFKA_OPTIONS)
    .option("topic", TOPIC_NAME)
    .save()
)

EXPECTED_EVENTS = {"spark-batch": batch_payload.count()}
print(f"Batch write complete: {EXPECTED_EVENTS['spark-batch']} Avro events produced to '{TOPIC_NAME}'.")

## 3.3. Read the topic back in batch mode and decode the Avro events
This confirms the batch write end to end: the bytes in Kafka are decoded with the registered schema.

In [ ]:
decoded_batch = from_confluent_avro(
    spark.read.format("kafka")
    .options(**KAFKA_OPTIONS)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .load()
)

this_run_batch = decoded_batch.where(F.col("run_id") == RUN_ID)
display(this_run_batch.orderBy("partition", "offset"))

## 3.4. Produce events with the Confluent Python client (batch)
The same topic can also be fed by an ordinary Python application. Here the `confluent-kafka` producer serializes events with Confluent's `AvroSerializer` and the registered schema. The delivery report shows the partition and offset assigned to each event.

* `acks=all` and `enable.idempotence=True` give the strongest delivery guarantee the producer offers. Retries cannot create duplicates inside the broker.
* `flush()` blocks until every event is acknowledged, so the cell finishes only after the data is safely in Kafka.

In [ ]:
import random

from confluent_kafka import Producer
from confluent_kafka.schema_registry.avro import AvroSerializer
from confluent_kafka.serialization import MessageField, SerializationContext, StringSerializer

customers = [
    {"customer_id": "C001", "name": "Jane Doe", "age": 29, "email": "jane.doe@example.com"},
    {"customer_id": "C002", "name": "Fatih", "age": 24, "email": "fatih@example.com"},
    {"customer_id": "C003", "name": "John Doe", "age": 35, "email": "john.doe@example.com"},
]
rng = random.Random(684)
now = datetime.now(timezone.utc)

python_events = []
for i, customer in enumerate(customers):
    for j, event_type in enumerate(["LOGIN", "PURCHASE", "LOGOUT"]):
        python_events.append({
            "event_id": str(uuid.uuid4()),
            "run_id": RUN_ID,
            "source": "python-producer",
            **customer,
            "event_type": event_type,
            "amount": round(rng.uniform(10, 250), 2) if event_type == "PURCHASE" else None,
            "event_time": now + timedelta(seconds=10 * i + j),
        })

avro_serializer = AvroSerializer(sr_client, AVRO_SCHEMA_STR)
key_serializer = StringSerializer("utf_8")
producer = Producer({**CLIENT_CONF, "acks": "all", "enable.idempotence": True, "client.id": "insy684-notebook"})

delivery_log = []


def on_delivery(err, msg):
    if err is not None:
        delivery_log.append({"status": "FAILED", "error": str(err)})
    else:
        delivery_log.append({"status": "OK", "partition": msg.partition(), "offset": msg.offset()})


value_ctx = SerializationContext(TOPIC_NAME, MessageField.VALUE)
key_ctx = SerializationContext(TOPIC_NAME, MessageField.KEY)
for event in python_events:
    producer.produce(
        topic=TOPIC_NAME,
        key=key_serializer(event["customer_id"], key_ctx),
        value=avro_serializer(event, value_ctx),
        on_delivery=on_delivery,
    )
    producer.poll(0)

remaining = producer.flush(30)
failed = [d for d in delivery_log if d["status"] != "OK"]
assert remaining == 0 and not failed, f"Delivery problems: remaining={remaining}, failed={failed}"

EXPECTED_EVENTS["python-producer"] = len(python_events)
print(f"Python producer delivered {len(delivery_log)} events. Partitions used: "
      f"{sorted({d['partition'] for d in delivery_log})}")

# 4. Stream Processing

Free Edition runs on serverless compute, which supports the `availableNow` trigger. With this trigger, a query processes everything that is new since its last checkpoint, in one or more micro-batches, and then stops. This is the same incremental model used in the Week 1 streaming notebooks. The checkpoint stores the progress, so the next run continues where the previous one stopped and nothing is processed twice.

## 4.1. Simulate an arriving event stream
Five JSON files are written to a volume. Each file stands for a burst of events arriving from an application. With `maxFilesPerTrigger = 1`, Spark treats every file as a new micro-batch, as in the course's `Spark_Streaming_Simulation` notebook.

In [ ]:
STREAM_FILES, EVENTS_PER_FILE = 5, 4
stream_customers = customers + [
    {"customer_id": "C004", "name": "Amira Haddad", "age": 31, "email": "amira.haddad@example.com"},
    {"customer_id": "C005", "name": "Luc Tremblay", "age": 42, "email": "luc.tremblay@example.com"},
]
rng = random.Random(2026)
start = datetime.now(timezone.utc).replace(microsecond=0)

Path(LANDING_PATH).mkdir(parents=True, exist_ok=True)
for file_no in range(STREAM_FILES):
    lines = []
    for k in range(EVENTS_PER_FILE):
        customer = rng.choice(stream_customers)
        event_type = rng.choice(["LOGIN", "PURCHASE", "PURCHASE", "LOGOUT"])
        lines.append(json.dumps({
            "event_id": str(uuid.uuid4()),
            **customer,
            "event_type": event_type,
            "amount": round(rng.uniform(5, 400), 2) if event_type == "PURCHASE" else None,
            "event_time": (start + timedelta(seconds=15 * (file_no * EVENTS_PER_FILE + k))).isoformat(),
        }))
    Path(f"{LANDING_PATH}/events_{file_no:02d}.json").write_text("\n".join(lines) + "\n")

print(f"Wrote {STREAM_FILES} files x {EVENTS_PER_FILE} events to {LANDING_PATH}")
print(Path(f"{LANDING_PATH}/events_00.json").read_text().splitlines()[0])

## 4.2. Write a Kafka sink for streaming queries (produce events as a stream)
The streaming query reads the landing folder, completes each record to the Avro schema, and writes it to the Kafka topic.

In [ ]:
landing_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("email", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("amount", DoubleType(), True),
    StructField("event_time", TimestampType(), False),
])

stream_events = (
    spark.readStream.schema(landing_schema)
    .option("maxFilesPerTrigger", 1)
    .json(LANDING_PATH)
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("source", F.lit("spark-stream"))
)

producer_query = (
    to_confluent_avro(stream_events)
    .writeStream.format("kafka")
    .options(**KAFKA_OPTIONS)
    .option("topic", TOPIC_NAME)
    .option("checkpointLocation", PRODUCER_CHECKPOINT)
    .trigger(availableNow=True)
    .queryName("stream_producer_to_kafka")
    .start()
)
producer_query.awaitTermination()


def rows_in(progress):
    # Newer runtimes return progress objects (read the attribute), older ones return plain dicts.
    value = getattr(progress, "numInputRows", None)
    if value is None and isinstance(progress, dict):
        value = progress.get("numInputRows")
    return value or 0


stream_batches = [rows_in(p) for p in producer_query.recentProgress if rows_in(p) > 0]
EXPECTED_EVENTS["spark-stream"] = STREAM_FILES * EVENTS_PER_FILE
print(f"Streaming producer finished: {len(stream_batches)} micro-batches, rows per batch = {stream_batches}")
print(f"Events sent to '{TOPIC_NAME}' by the stream: {EXPECTED_EVENTS['spark-stream']} (checked in section 5)")

## 4.3. Read a stream from Kafka and land it in a Delta table
This is the consumer side. The stream reads the topic, decodes the Avro events with the registered schema, and appends them to a bronze Delta table.

* `startingOffsets = earliest` applies only to the first run. After that, the checkpoint decides where reading resumes.
* `failOnDataLoss = false` lets the query continue if Confluent has already deleted old messages under the topic's retention policy (7 days by default).

In [ ]:
kafka_stream = (
    spark.readStream.format("kafka")
    .options(**KAFKA_OPTIONS)
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

consumer_query = (
    from_confluent_avro(kafka_stream)
    .withColumn("ingested_at", F.current_timestamp())
    .writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", CONSUMER_CHECKPOINT)
    .trigger(availableNow=True)
    .queryName("kafka_to_bronze")
    .toTable(BRONZE_TABLE)
)
consumer_query.awaitTermination()

consumed = sum(rows_in(p) for p in consumer_query.recentProgress)
print(f"Streaming consumer finished: {consumed} new Kafka records read in this run.")
display(spark.table(BRONZE_TABLE).where(F.col("run_id") == RUN_ID).orderBy("event_time"))

## 4.4. Stream analytics on the landed events
Once events are in Delta they can be queried with SQL. The first query summarizes every event in the table by producer and event type. The second groups this run's purchases into one minute event-time windows, the same windowing idea used in the Week 1 Kafka notebook.

In [ ]:
%sql
SELECT source,
       event_type,
       COUNT(*)              AS events,
       ROUND(SUM(amount), 2) AS revenue_cad
FROM workspace.default.customer_events_bronze
GROUP BY source, event_type
ORDER BY source, event_type

In [ ]:
purchases_per_minute = (
    spark.table(BRONZE_TABLE)
    .where((F.col("run_id") == RUN_ID) & (F.col("event_type") == "PURCHASE"))
    .groupBy(F.window("event_time", "1 minute"))
    .agg(F.count("*").alias("purchases"), F.round(F.sum("amount"), 2).alias("revenue_cad"))
    .select(F.col("window.start").alias("window_start"), F.col("window.end").alias("window_end"),
            "purchases", "revenue_cad")
    .orderBy("window_start")
)
display(purchases_per_minute)

# 5. End-to-end verification
The checks below compare what each producer sent in this run with what the streaming consumer landed in Delta. They also confirm that every event carries the registered schema id and that no event was duplicated.

In [ ]:
landed = spark.table(BRONZE_TABLE).where(F.col("run_id") == RUN_ID)
observed = {r["source"]: r["n"] for r in landed.groupBy("source").agg(F.countDistinct("event_id").alias("n")).collect()}
total_rows = landed.count()
distinct_ids = landed.select("event_id").distinct().count()
schema_ids = sorted(r["schema_id"] for r in landed.select("schema_id").distinct().collect())

checks = [(f"{src} events landed", EXPECTED_EVENTS[src], observed.get(src, 0)) for src in EXPECTED_EVENTS]
checks += [
    ("Total events landed", sum(EXPECTED_EVENTS.values()), distinct_ids),
    ("Duplicate events (rows minus distinct ids)", 0, total_rows - distinct_ids),
    ("Schema ids found in landed events", [SCHEMA_ID], schema_ids),
]

print(f"{'Check':<45}{'Expected':>12}{'Observed':>12}   Result")
for name, expected, actual in checks:
    print(f"{name:<45}{str(expected):>12}{str(actual):>12}   {'PASS' if expected == actual else 'FAIL'}")

assert all(expected == actual for _, expected, actual in checks), "At least one verification check failed."
print("\nAll checks passed: batch and streaming events were produced to Kafka and processed by Spark.")

# 6. Summary

**Kafka (Confluent Cloud).** A Basic cluster hosts the topic `databricks_test`. A new Avro schema, `CustomerEvent`, is registered under the subject `databricks_test-value`. The topic works as an append-only, partitioned log. Records with the same key (the customer id) go to the same partition, so the events of each customer stay in order.

**Spark (Databricks).** Events were produced to the topic in two ways:
* **Batch:** a Spark DataFrame write (section 3.2) and the `confluent-kafka` Python producer (section 3.4).
* **Streaming:** a Structured Streaming query fed by arriving files (section 4.2).

The topic was then consumed in batch mode (section 3.3) and as a stream into a Delta table (section 4.3).

**Batch compared with streaming.** A batch query reads a bounded snapshot of the topic and stops. A streaming query treats the topic as unbounded input. Its checkpoint records which offsets have already been processed, so each run handles only new events. With `availableNow`, the stream runs incrementally on demand. This is the most cost-efficient pattern on serverless compute, and it can be scheduled as a job when lower latency is needed.

**Delivery guarantees.** Kafka sinks in Spark deliver at least once, so a retry can write an event twice. The `event_id` field makes duplicates easy to detect and remove later. Section 5 confirms that this run contains none.

# 7. Clean-up
As requested in the course announcement, every resource created for this assignment is deleted once the screenshots are taken and this notebook is exported, so no free credits are used up and nothing is charged.

* **Databricks:** uncomment and run the cell below. It stops any active stream and removes the Delta table, the volume (checkpoints and landing files) and the `confluent` secret scope.
* **Confluent Cloud:** delete the cluster (Cluster settings > Delete cluster), which also deletes the topic, then delete both API keys and the `databricks_test-value` schema subject.

In [ ]:
# from databricks.sdk import WorkspaceClient
#
# for q in spark.streams.active:
#     q.stop()
# spark.sql(f"DROP TABLE IF EXISTS {BRONZE_TABLE}")
# spark.sql(f"DROP VOLUME IF EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
# WorkspaceClient().secrets.delete_scope(scope=SECRET_SCOPE)
# print("Databricks resources removed: table, volume and secret scope.")